# Fraud Analytics: detecting synthetic travel-expense fraud
**Kyaw Thiha (Kendrick) · ITCY514, Mahidol University**

The Aster Field Services case compares rules, supervised learning, association mining and graph analysis. All people, businesses, accounts, transactions and evaluation labels are fictitious. Peer-review exercises are outside this public release.

Run from the repository root or the `final/` directory after installing `final/requirements-notebook.txt`. The companion `tools/` directory is required. Reruns write to `final/notebook_run/` and preserve the committed evidence.


## 1. Set up the experiment
The random seed, features, policy settings and model logic come from the same implementation used for the report.
Dependencies are pinned in `requirements.txt`. If needed, install them once in a separate cell with
`%pip install -r requirements.txt`, then restart the kernel.

In [ ]:
from pathlib import Path
import importlib.util
import json
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT = Path.cwd()
if not (PROJECT / "tools/project_core.py").exists():
    PROJECT = PROJECT / "final"
if not (PROJECT / "tools/project_core.py").exists():
    raise FileNotFoundError("Open from the repository root or final directory; keep the companion tools folder.")

spec = importlib.util.spec_from_file_location("aster_core", PROJECT / "tools/project_core.py")
core = importlib.util.module_from_spec(spec)
spec.loader.exec_module(core)
sys.path.insert(0, str(PROJECT / "tools"))
core.ROOT = PROJECT / "notebook_run"
for folder in ["data/landing", "data/production", "private", "reports", "models"]:
    (core.ROOT / folder).mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_colwidth", 55)
print(f"Seed: {core.SEED} | Output folder: {core.ROOT.name}")

## 2. Generate the fictional case and check the data
Aster reimburses business travel. The case includes inflated hotel expenses, repeated reimbursements,
approval abuse and coordinated employee–merchant activity. Legitimate expensive trips, authorized exceptions
and disclosed relationships provide alternative explanations.

The generator creates five source tables and a separate private answer key. Invalid rows are quarantined,
not treated as fraud. Row counts and numeric amounts must reconcile before modelling.

In [ ]:
data = core.generate()
valid, controls = core.prepare(data)

display(pd.DataFrame({"Source table": data.keys(), "Rows": [len(t) for t in data.values()]}))
display(pd.DataFrame({
    "Stage": ["Raw claims", "Valid claims", "Quarantine"],
    "Rows": [controls["raw_claims"], controls["production_claims"], controls["quarantined_claims"]],
    "Numeric amount (THB)": [controls["raw_numeric_thb"], controls["production_thb"], controls["quarantine_numeric_thb"]],
}).round(2))
assert controls["row_difference"] == 0
assert abs(controls["amount_difference_thb"]) <= 0.01
print("Reconciled. One nonnumeric amount is retained in quarantine and excluded from numeric totals.")

## 3. Build predictors and preserve time order
**Scoring point:** after approval, before reimbursement. Price ratios, approval behaviour and past activity
are available at this point. Receipt reuse, claim counts and merchant shares look backward only.
Employee IDs, merchant IDs, private fraud labels, scheme names and true excess payments do not enter the classifiers.

| Partition | Months in fictional 2025 | Purpose |
|---|---|---|
| Train | January–June | Fit models and preprocessing |
| Validation | July–August | Choose alert thresholds and the selected model |
| Test | September–October | Final comparison after choices are fixed |
| Later period | November–December | Check performance under subtler simulated inflation |

Hotel allowance is THB 9,000 per **approved night**; documented exceptions can raise the allowance.
City price benchmarks are fictional reference settings, not scraped prices.

In [ ]:
features = core.engineer(valid, data)
partition_order = list(core.SPLITS)
split_sizes = features.groupby("split").size().reindex(partition_order)
display(split_sizes.rename_axis("Partition").to_frame("Claims"))

display(pd.DataFrame({
    "Numeric predictors": pd.Series(core.NUMERIC),
    "Categorical predictors": pd.Series(core.CATEGORICAL),
}).fillna(""))

**Earlier lectures: distributions and comparisons.** The following training-only profiles use hotel
price ratios, quartiles, MAD and robust scores. Employee and monthly comparisons are descriptive.
A robust score above 3.5 is an outlier lead, not a fraud label or a p-value. Each month has 400 claims
by generator design. No formal significance-test or Benford conclusion is claimed.

In [ ]:
from graph_analysis import profile_training

profile = profile_training(features, core.ROOT)
display(pd.Series({key: value for key, value in profile.items() if key != "interpretation"}, name="Training profile").to_frame())
display(pd.read_csv(core.ROOT / "reports/training_monthly_profile.csv").round(3))
display(pd.read_csv(core.ROOT / "reports/training_peer_profile.csv").head(5).round(3))

## 4. Train the models and fix the thresholds
**Logistic regression:** standardized numeric inputs, one-hot categories, C=1 and balanced class weights.
**Random Forest:** 240 trees, maximum depth 9, minimum leaf size 8 and balanced subsample weights.

Both fit on January–June only. A predefined threshold grid from 0.05 to 0.95 is evaluated on validation F1;
ties use precision, then the higher threshold. Validation F1 also selects the model for discussion.
Models are not refitted after selection. The test and later period do not influence these choices.
The baseline flags prior receipt reuse, self-approval, over-allowance amounts or an undisclosed account match.

In [ ]:
predictions, metric_rows, curves, selected_model, thresholds = core.train_evaluate(features)
metrics = pd.DataFrame(metric_rows)
metric_columns = ["model", "tp", "fp", "fn", "precision", "recall", "f1", "average_precision"]

print(f"Selected on validation: {selected_model}")
display(pd.Series(thresholds, name="Fixed alert threshold").to_frame())
display(metrics.loc[metrics["split"].eq("validation"), metric_columns].round(3))

## 5. Compare the models on the held-out test
All three methods are evaluated on the same 800 claims. Precision measures how many alerts are correct;
recall measures how much planted fraud is detected. F1 balances these two measures.
Average precision (AP) summarizes score ranking; the binary rules have no comparable ranked AP.
The all-legitimate accuracy baseline is 92.5% and detects no fraud, so accuracy is not the headline metric.

In [ ]:
test_metrics = metrics.loc[metrics["split"].eq("test")].set_index("model")
display(test_metrics[["tp", "fp", "fn", "tn", "alerts", "precision", "recall", "f1", "average_precision"]].round(3))

fig, ax = plt.subplots(figsize=(7, 4))
for slug, label in [("lr", "Logistic regression"), ("rf", "Random Forest")]:
    curve = curves[f"{slug}_test"]
    ax.plot(curve["recall"], curve["precision"], label=label)
rules = test_metrics.loc["Existing rules"]
ax.scatter(rules["recall"], rules["precision"], marker="x", color="black", label="Existing rules")
ax.axhline(60 / 800, color="grey", linestyle="--", label="Test prevalence (7.5%)")
ax.set(xlabel="Recall", ylabel="Precision", xlim=(0, 1), ylim=(0, 1.03), title="Test precision–recall performance")
ax.legend(loc="lower left")
fig.tight_layout()
plt.show()

**Finding:** Random Forest catches 44 of 60 planted fraud claims with 12 false alarms.
Logistic regression catches four more fraud claims but generates nine more false alarms.
The rules catch 24 with 35 false alarms. Random Forest was chosen on validation performance;
these test results describe the trade-off, rather than selecting a new winner from the test.

## 6. Explain the model signals
Logistic coefficients describe conditional associations. For numeric inputs, the odds ratios represent a
one-training-standard-deviation change, including binary inputs; they are not causal or native-unit effects.
Forest permutation importance measures the decrease in **validation AP** when a predictor is shuffled.

In [ ]:
coefficients = pd.read_csv(core.ROOT / "reports/logistic_coefficients.csv")
top_coefficients = coefficients.assign(magnitude=coefficients["coefficient"].abs()).nlargest(6, "magnitude")
display(top_coefficients.drop(columns="magnitude").round(3))

importance = pd.read_csv(core.ROOT / "reports/rf_permutation_importance.csv")
display(importance.head(6).round(3))

## 7. Discover recurring relationships with FP-Growth
Each training claim is a basket of employee, merchant and approver. Discovery uses training data only:
minimum support 0.004, confidence 0.20, lift 2 and maximum itemset length 3.
No fraud labels enter the baskets. The discovered rules are then checked for persistence in unseen periods.

High lift can result from rare merchants or repeated legitimate business. The disclosed E020–V036 relationship
is a negative control: a strong association alone does not establish collusion.

In [ ]:
association_rows = core.associations(features)
associations = pd.DataFrame(association_rows)
association_columns = ["antecedents", "consequent", "train_count", "train_confidence", "train_lift", "test_count"]
print(f"Qualifying training rules: {len(associations)}")
display(associations[association_columns].round(3))

## 8. Apply EP14: observed networks, communities and paths
The graph uses training claims only. Employees and merchants are different node types. Each edge represents
observed expense affiliation and retains claim counts, amounts, approver IDs and source claim IDs.
Degree counts unique counterparties. Unweighted betweenness summarizes shortest-path position.
Greedy modularity uses claim-count weights and resolution 1. These descriptive outputs do not establish guilt.

The displayed focus uses merchants from training FP-Growth rules and edges with at least three claims.
It omits other observed links. Community membership is calculated on the full training graph, not this focus.

In [ ]:
from graph_analysis import analyze_graph
from build_graph_outputs import graph_figure
from IPython.display import Image

graph = analyze_graph(features, core.ROOT, association_rows)
display(pd.Series({key: graph[key] for key in [
    "training_claims", "employees", "merchants", "nodes", "edges", "components",
    "bipartite_density", "community_count", "weighted_modularity",
]}, name="Observed training network").to_frame().round(3))

figure_path = core.ROOT / "reports/figures/graph_relationships.png"
figure_path.parent.mkdir(exist_ok=True)
graph_figure(graph, figure_path)
display(Image(filename=str(figure_path), width=850))
display(pd.read_csv(core.ROOT / "reports/graph_nodes.csv").query("node_id in ['E009','E037','E065','E020','V031','V032','V036']").round(4))
display(pd.read_csv(core.ROOT / "reports/graph_paths.csv")[[
    "employee_a", "via_merchant", "employee_b", "left_claim_count", "right_claim_count",
]])

**Interpretation:** the graph contains 147 observed nodes, 1,703 distinct links and nine communities.
E009, E037, E065, V031 and V032 share community 7, which contains 12 nodes rather than only those five entities.
The disclosed E020–V036 pair forms a separate two-node community. A community therefore does not imply fraud.

The path **E037–V032–E065** shows shared merchant use, supported by 11 and seven source claims on its legs.
It does not establish direct employee ties, ownership or money flow. Labels and model scores are excluded.
No graph predictor is added to the classifiers, so the supervised performance figures remain the same.
The companion exports retain the full edge list, node measures, communities and path provenance.

## 9. Investigate a false alarm, a missed claim and an additional detection
Private synthetic labels are opened here to evaluate errors; investigators would need independent documents
to establish what happened in a real case. A score is an uncalibrated model output, not a real-world fraud probability.

In [ ]:
test = predictions.loc[predictions["split"].eq("test")].copy()
examples = {
    "Additional detection": "FC-03568",
    "False alarm": "FC-03789",
    "Missed fraud": "FC-03578",
    "Authorized exception": "FC-03490",
}
case_examples = test.set_index("claim_id").loc[list(examples.values()),
    ["amount_thb", "rule_alert", "rf_score", "rf_alert", "is_fraud", "scheme"]].copy()
case_examples.insert(0, "Review example", list(examples))
display(case_examples.round(3))

review_queue = test.loc[test["selected_alert"].eq(1)].drop_duplicates("claim_id")
print(f"Unique test alerts: {len(review_queue)}")
print(f"Gross review exposure: THB {review_queue['amount_thb'].sum():,.2f} — not confirmed loss.")

**Follow-up:** request hotel folios and booking evidence for additional detections; check legitimate travel
context before escalating a false alarm; inspect authorization and trip records for the approval-abuse miss.
Confirmed intent cannot be inferred from an alert alone.

## 10. Check the later period and the experiment controls
November–December simulates subtler hotel inflation. Thresholds stay fixed. This checks performance on later
records with familiar entities; it does not establish generalization to new employees or wholly new fraud schemes.

In [ ]:
later = metrics.loc[metrics["split"].eq("out_of_time")].set_index("model")
display(later[["tp", "fp", "fn", "precision", "recall", "f1", "average_precision"]].round(3))

checks = core.validate(features, predictions, controls)
print(f"Integrity and leakage checks passed: {sum(c['pass_'] for c in checks)}/{len(checks)}")
display(pd.DataFrame(checks)[["check", "pass_"]].rename(columns={"pass_": "Passed"}))

**Finding:** Random Forest recall falls from 73.3% to 58.3% in the later period, despite precision staying
near 80%. A useful model still needs monitoring and investigation of missed cases.

## 11. Conclusions and the peer exchange
- Supervised models improve this synthetic experiment over the existing rules, with a measurable precision–recall trade-off.
- FP-Growth provides relationship leads and a legitimate counterexample; it is not a fraud verdict.
- EP14 adds executed centrality, community and path analysis. These are descriptive graph leads, with source provenance.
- One synthetic generator, one seed, overlapping entities and uncalibrated scores limit real-world claims.
- AI assisted code and writing; Python computes the results. No external AI API is called by this notebook.

Peer review is outside this portfolio release. No independent peer outcome is claimed.

Reruns export metrics, synthetic evaluation labels and model artifacts under `notebook_run/`.
